# Posit API Integration Test — Comprehensive

Exercises every prod-mode API endpoint and data flow. Run top-to-bottom against a fresh server started with `./start.sh --prod`.

**Coverage:**
- Health check, empty state verification
- Stream CRUD (create, list, update, delete)
- Stream configuration (PENDING → READY)
- Bankroll (read + set)
- Aggregate market values (set, list, delete)
- Snapshot ingestion (with optional `market_value` field)
- Pipeline queries (dimensions, time series)
- Block table (list, manual create, update)
- Transform configuration (inspect, update)
- Client WebSocket (snapshot frames, market value frames)
- LLM investigation (SSE — skipped if no API key)
- WS position subscription
- Error cases
- Cleanup

In [ ]:
import httpx
import asyncio
import json
import websockets
from datetime import datetime, timedelta, timezone
from pprint import pprint

# ── Target: uncomment ONE of these ──────────────────────────────────────
BASE = "localhost:8000";         SECURE = False   # local dev
# BASE = "posit-admin.up.railway.app"; SECURE = True  # production (Railway)

HTTP_SCHEME = "https" if SECURE else "http"
WS_SCHEME   = "wss"   if SECURE else "ws"

client = httpx.Client(base_url=f"{HTTP_SCHEME}://{BASE}/", timeout=30.0)

# Client WS auth (must match CLIENT_WS_API_KEY on the server)
API_KEY = "lKQ4riHaZCFjtNZB_qrzVAA9X5wl_iI-SHZr5Xw8ESw"
CLIENT_WS_URL = f"{WS_SCHEME}://{BASE}/ws/client?api_key={API_KEY}"
WS_URL = f"{WS_SCHEME}://{BASE}/ws"

print("Target:", f"{HTTP_SCHEME}://{BASE}")
print("Client WS:", CLIENT_WS_URL)
print("Pipeline WS:", WS_URL)

## 1. Health Check

In [41]:
r = client.get("/api/health")
print(r.status_code, r.json())

200 {'status': 'ok'}


## 2. Verify Empty State (prod mode)

In prod mode, no streams or pipeline data should exist yet.

In [ ]:
# In prod mode, only the pre-configured __test__ stream should exist
r = client.get("/api/streams")
streams = r.json()["streams"]
print(f"Streams at startup: {len(streams)}")
for s in streams:
    print(f"  {s['stream_name']:12s}  status={s['status']}")

assert len(streams) == 1, f"Expected 1 stream (__test__), got {len(streams)}"
assert streams[0]["stream_name"] == "__test__"
assert streams[0]["status"] == "READY"

# No pipeline data yet (no snapshots ingested for real streams)
r = client.get("/api/pipeline/dimensions")
dims = r.json()["dimensions"]
print(f"\nPipeline dimensions: {len(dims)}")

# No blocks yet
r = client.get("/api/blocks")
blocks = r.json()["blocks"]
print(f"Blocks: {len(blocks)}")

# No market values
r = client.get("/api/market-values")
mvs = r.json()["entries"]
print(f"Market values: {len(mvs)}")
print("\n✓ Empty state verified")

## 3. Test Data Setup

Define timestamps, symbols, and expiries for a multi-dimensional test scenario.
We'll use two symbols (BTC, ETH) and two expiries to exercise cross-dimension behavior.

In [ ]:
NOW = datetime.now(timezone.utc).replace(tzinfo=None)
NOW_ISO = NOW.isoformat()

SYMBOLS = ["BTC", "ETH"]
EXPIRY_1 = NOW + timedelta(days=1)
EXPIRY_2 = NOW + timedelta(days=7)
EXPIRY_1_ISO = EXPIRY_1.isoformat()
EXPIRY_2_ISO = EXPIRY_2.isoformat()

# Event timestamps for the events stream (5 events, spaced 4h apart)
NUM_EVENTS = 5
EVENT_STARTS = [NOW + timedelta(hours=i * 4) for i in range(NUM_EVENTS)]

print(f"NOW:      {NOW_ISO}")
print(f"EXPIRY_1: {EXPIRY_1_ISO}  (1 day)")
print(f"EXPIRY_2: {EXPIRY_2_ISO}  (7 days)")
print(f"SYMBOLS:  {SYMBOLS}")
print(f"EVENTS:   {NUM_EVENTS} events, 4h spacing")

## 4. Stream CRUD — Create

Create three streams mirroring the mock scenario: `rv`, `mean_iv`, `events`.
Each starts in PENDING status until admin configures pipeline parameters.

In [ ]:
STREAMS = [
    {"stream_name": "rv",      "key_cols": ["symbol", "expiry"]},
    {"stream_name": "mean_iv", "key_cols": ["symbol", "expiry"]},
    {"stream_name": "events",  "key_cols": ["symbol", "expiry", "event_id"]},
]

for s in STREAMS:
    r = client.post("/api/streams", json=s)
    assert r.status_code == 201, f"Create {s['stream_name']} failed: {r.status_code} {r.text}"
    data = r.json()
    assert data["status"] == "PENDING"
    print(f"  {s['stream_name']:10s}  status={data['status']}  key_cols={data['key_cols']}")

print(f"\n✓ {len(STREAMS)} streams created (all PENDING)")

## 4b. Stream CRUD — List & Update

Verify listing, then rename a stream and verify the rename took effect.

In [ ]:
# List all streams (should include __test__ + 3 new ones)
r = client.get("/api/streams")
assert r.status_code == 200
all_streams = r.json()["streams"]
stream_names = [s["stream_name"] for s in all_streams]
print(f"All streams ({len(all_streams)}):", stream_names)
assert "__test__" in stream_names
assert "rv" in stream_names
assert "mean_iv" in stream_names
assert "events" in stream_names

# Update: rename "rv" → "rv_temp", then rename back
r = client.patch("/api/streams/rv", json={"stream_name": "rv_temp"})
assert r.status_code == 200
assert r.json()["stream_name"] == "rv_temp"
print("\nRenamed rv → rv_temp")

# Verify it's renamed in listing
r = client.get("/api/streams")
names = [s["stream_name"] for s in r.json()["streams"]]
assert "rv_temp" in names and "rv" not in names

# Rename back
r = client.patch("/api/streams/rv_temp", json={"stream_name": "rv"})
assert r.status_code == 200
assert r.json()["stream_name"] == "rv"
print("Renamed rv_temp → rv (restored)")
print("\n✓ Stream list & update work correctly")

## 5. Configure Streams (admin)

Set pipeline-facing parameters (scale, offset, exponent, block config) to move each stream to READY.
Each stream uses different block parameters to exercise the full config surface.

In [ ]:
CONFIGS = {
    "rv": {
        "scale": 1.0,
        "offset": 0.0,
        "exponent": 2,
        "block": {"annualized": True},
    },
    "mean_iv": {
        "scale": 1.0,
        "offset": 0.0,
        "exponent": 2,
        "block": {
            "annualized": True,
            "aggregation_logic": "offset",
            "size_type": "relative",
            "decay_end_size_mult": 0.0,
            "decay_rate_prop_per_min": 0.01,
            "var_fair_ratio": 2.0,
        },
    },
    "events": {
        "scale": 0.01,
        "offset": 0.0,
        "exponent": 2,
        "block": {
            "annualized": False,
            "aggregation_logic": "offset",
            "temporal_position": "static",
            "decay_end_size_mult": 0.0,
            "decay_rate_prop_per_min": 0.01,
            "var_fair_ratio": 3.0,
        },
    },
}

for name, cfg in CONFIGS.items():
    r = client.post(f"/api/streams/{name}/configure", json=cfg)
    assert r.status_code == 200, f"Configure {name} failed: {r.status_code} {r.text}"
    data = r.json()
    assert data["status"] == "READY"
    print(f"  {name:10s}  status={data['status']}  scale={data['scale']}  exponent={data['exponent']}")

print(f"\n✓ {len(CONFIGS)} streams configured (all READY)")

## 6. Bankroll

Read the default bankroll, then set a new value. Verify the round-trip.

In [ ]:
# Read current bankroll
r = client.get("/api/config/bankroll")
assert r.status_code == 200
print(f"Current bankroll: {r.json()['bankroll']}")

# Set bankroll
BANKROLL = 100_000
r = client.patch("/api/config/bankroll", json={"bankroll": BANKROLL})
assert r.status_code == 200
data = r.json()
assert data["bankroll"] == BANKROLL
print(f"Set bankroll: {data['bankroll']}")
print(f"Pipeline rerun: {data['pipeline_rerun']}")

# Verify read-back
r = client.get("/api/config/bankroll")
assert r.json()["bankroll"] == BANKROLL
print(f"\n✓ Bankroll set to {BANKROLL}")

## 7. Aggregate Market Values

Set aggregate market values for each symbol/expiry pair. These represent the market-implied
total vol that the pipeline uses to compute edge (fair − market). The dirty flag triggers
a coalesced pipeline rerun on the next WS tick.

In [ ]:
# PUT /api/market-values — batch set
MV_ENTRIES = [
    {"symbol": "BTC", "expiry": EXPIRY_1_ISO, "total_vol": 0.55},
    {"symbol": "BTC", "expiry": EXPIRY_2_ISO, "total_vol": 0.48},
    {"symbol": "ETH", "expiry": EXPIRY_1_ISO, "total_vol": 0.70},
    {"symbol": "ETH", "expiry": EXPIRY_2_ISO, "total_vol": 0.62},
]

r = client.put("/api/market-values", json={"entries": MV_ENTRIES})
assert r.status_code == 200
data = r.json()
print(f"Market values set: {len(data['entries'])} entries")
for e in data["entries"]:
    print(f"  {e['symbol']:>5s} / {e['expiry'][:10]}  total_vol={e['total_vol']}")

# GET /api/market-values — verify round-trip
r = client.get("/api/market-values")
assert r.status_code == 200
assert len(r.json()["entries"]) == len(MV_ENTRIES)

# DELETE one entry and verify
r = client.delete(f"/api/market-values/ETH/{EXPIRY_2_ISO}")
assert r.status_code == 200
assert r.json()["deleted"] is True
print(f"\nDeleted ETH/{EXPIRY_2_ISO[:10]}")

r = client.get("/api/market-values")
remaining = r.json()["entries"]
assert len(remaining) == len(MV_ENTRIES) - 1
print(f"Remaining: {len(remaining)} entries")

# Re-add the deleted entry for downstream tests
r = client.put("/api/market-values", json={"entries": MV_ENTRIES})
assert r.status_code == 200
print(f"\nRestored all {len(MV_ENTRIES)} entries")
print("✓ Market values CRUD works")

## 8. Snapshot Ingestion

Send snapshot data for all three streams. Each snapshot row includes `timestamp`, `raw_value`,
the stream's key columns, and optionally `market_value`.

This triggers `rerun_pipeline()` and the WS ticker restarts — the UI should update.

In [ ]:
# -- rv stream (multi-symbol, multi-expiry, with market_value) --
rv_rows = []
for sym in SYMBOLS:
    for exp_iso in [EXPIRY_1_ISO, EXPIRY_2_ISO]:
        rv_val = 0.45 if sym == "BTC" else 0.60
        rv_rows.append({
            "timestamp": NOW_ISO,
            "symbol": sym,
            "expiry": exp_iso,
            "raw_value": rv_val,
            "market_value": rv_val + 0.05,  # market implies slightly higher vol
        })

r = client.post("/api/snapshots", json={"stream_name": "rv", "rows": rv_rows})
assert r.status_code == 200
data = r.json()
assert data["pipeline_rerun"] is True
print(f"rv:      {data['rows_accepted']} rows, pipeline_rerun={data['pipeline_rerun']}")

# -- mean_iv stream (multi-symbol, multi-expiry) --
iv_rows = []
for sym in SYMBOLS:
    for exp_iso in [EXPIRY_1_ISO, EXPIRY_2_ISO]:
        iv_val = 0.50 if sym == "BTC" else 0.65
        iv_rows.append({
            "timestamp": NOW_ISO,
            "symbol": sym,
            "expiry": exp_iso,
            "raw_value": iv_val,
        })

r = client.post("/api/snapshots", json={"stream_name": "mean_iv", "rows": iv_rows})
assert r.status_code == 200
data = r.json()
assert data["pipeline_rerun"] is True
print(f"mean_iv: {data['rows_accepted']} rows, pipeline_rerun={data['pipeline_rerun']}")

# -- events stream (multi-symbol, only EXPIRY_1 for events) --
event_rows = []
for sym in SYMBOLS:
    for i in range(NUM_EVENTS):
        event_rows.append({
            "timestamp": NOW_ISO,
            "symbol": sym,
            "expiry": EXPIRY_1_ISO,
            "event_id": f"event_{i}",
            "raw_value": [2.5, 3.1, 1.8, 4.0, 2.0][i],
            "start_timestamp": EVENT_STARTS[i].isoformat(),
        })

r = client.post("/api/snapshots", json={"stream_name": "events", "rows": event_rows})
assert r.status_code == 200
data = r.json()
assert data["pipeline_rerun"] is True
print(f"events:  {data['rows_accepted']} rows, pipeline_rerun={data['pipeline_rerun']}")
print(f"\n✓ All snapshots ingested, pipeline running")

## 9. Verify Streams & Pipeline State

Confirm all streams are READY and the pipeline has produced data.

In [ ]:
# Verify streams
r = client.get("/api/streams")
assert r.status_code == 200
for s in r.json()["streams"]:
    print(f"  {s['stream_name']:12s}  status={s['status']}")
assert all(s["status"] == "READY" for s in r.json()["streams"])

# Verify pipeline dimensions — should have our symbol/expiry pairs
r = client.get("/api/pipeline/dimensions")
assert r.status_code == 200
dims = r.json()["dimensions"]
print(f"\nPipeline dimensions: {len(dims)}")
for d in dims:
    print(f"  {d['symbol']:>5s} / {d['expiry'][:10]}")
assert len(dims) >= 2, f"Expected at least 2 dimensions, got {len(dims)}"

print("\n✓ All streams READY, pipeline has data")

## 10. Pipeline Time Series

Fetch the full block-level and aggregated time series for a specific symbol/expiry.
Validates the response shape matches `PipelineTimeSeriesResponse`.

In [ ]:
# Use the first dimension returned
dim = dims[0]
r = client.get("/api/pipeline/timeseries", params={"symbol": dim["symbol"], "expiry": dim["expiry"]})
assert r.status_code == 200
ts_data = r.json()

print(f"Time series for {ts_data['symbol']} / {ts_data['expiry'][:10]}:")
print(f"  Blocks: {len(ts_data['blocks'])}")
for b in ts_data["blocks"]:
    print(f"    {b['blockName']:30s}  space={b['spaceId']:12s}  points={len(b['timestamps'])}")

agg = ts_data["aggregated"]
print(f"  Aggregated points: {len(agg['timestamps'])}")

# Validate aggregated shape has all expected fields
AGG_FIELDS = ["timestamps", "totalFair", "totalMarketFair", "edge", "smoothedEdge",
              "var", "smoothedVar", "rawDesiredPosition", "smoothedDesiredPosition"]
for field in AGG_FIELDS:
    assert field in agg, f"Missing aggregated field: {field}"

# Check currentDecomposition
decomp = ts_data["currentDecomposition"]
print(f"  Current decomposition blocks: {len(decomp['blocks'])}")
print(f"  Aggregated snapshot keys: {list(decomp['aggregated'].keys())}")
if decomp.get("aggregateMarketValue"):
    print(f"  Aggregate market value: {decomp['aggregateMarketValue']}")

print("\n✓ Pipeline time series response validated")

## 11. Block Table

List all blocks from the current pipeline run. Each block maps to a stream × key_cols combination.

In [ ]:
r = client.get("/api/blocks")
assert r.status_code == 200
blocks = r.json()["blocks"]
print(f"Blocks in pipeline: {len(blocks)}")

# Validate BlockRowResponse shape
BLOCK_FIELDS = [
    "block_name", "stream_name", "symbol", "expiry", "space_id", "source",
    "annualized", "size_type", "aggregation_logic", "temporal_position",
    "decay_end_size_mult", "decay_rate_prop_per_min", "var_fair_ratio",
    "scale", "offset", "exponent", "target_value", "raw_value",
]
for b in blocks:
    for field in BLOCK_FIELDS:
        assert field in b, f"Block {b.get('block_name', '?')} missing field: {field}"

# Print summary
for b in blocks:
    print(
        f"  {b['block_name']:35s}  {b['stream_name']:10s}  {b['symbol']:>5s}  "
        f"space={b['space_id']:12s}  source={b['source']:6s}  "
        f"raw={b['raw_value']:.4f}  target={b['target_value']:.6f}"
    )

# All should be "stream" source (no manual blocks yet)
sources = set(b["source"] for b in blocks)
assert sources == {"stream"}, f"Expected only 'stream' source, got {sources}"
print(f"\n✓ {len(blocks)} blocks, all source=stream")

## 12. Manual Block Creation

Create a manual block (discretionary view). This creates a stream, configures it,
ingests snapshot data, and re-runs the pipeline in one atomic operation.

In [ ]:
MANUAL_BLOCK = {
    "stream_name": "manual_btc_vol_spike",
    "key_cols": ["symbol", "expiry"],
    "scale": 1.0,
    "offset": 0.0,
    "exponent": 2,
    "block": {
        "annualized": True,
        "aggregation_logic": "offset",
        "temporal_position": "shifting",
        "size_type": "fixed",
        "decay_end_size_mult": 0.5,
        "decay_rate_prop_per_min": 0.005,
        "var_fair_ratio": 1.5,
    },
    "snapshot_rows": [
        {
            "timestamp": NOW_ISO,
            "symbol": "BTC",
            "expiry": EXPIRY_1_ISO,
            "raw_value": 0.08,
            "market_value": 0.05,
        },
    ],
}

r = client.post("/api/blocks", json=MANUAL_BLOCK)
assert r.status_code == 201, f"Create manual block failed: {r.status_code} {r.text}"
mb = r.json()
print(f"Manual block created:")
print(f"  block_name:  {mb['block_name']}")
print(f"  source:      {mb['source']}")
print(f"  symbol:      {mb['symbol']}")
print(f"  raw_value:   {mb['raw_value']}")
print(f"  var_fair_ratio: {mb['var_fair_ratio']}")
assert mb["source"] == "manual"

# Verify it appears in block listing
r = client.get("/api/blocks")
all_blocks = r.json()["blocks"]
manual_blocks = [b for b in all_blocks if b["source"] == "manual"]
print(f"\nBlock table now has {len(all_blocks)} blocks ({len(manual_blocks)} manual)")
assert len(manual_blocks) >= 1
print("✓ Manual block created and visible in block table")

## 13. Block Update

Update the manual block's parameters and snapshot, then verify the change propagated.

In [ ]:
# Update the manual block: change var_fair_ratio and snapshot value
UPDATE_PAYLOAD = {
    "block": {
        "annualized": True,
        "aggregation_logic": "offset",
        "temporal_position": "shifting",
        "size_type": "fixed",
        "decay_end_size_mult": 0.5,
        "decay_rate_prop_per_min": 0.005,
        "var_fair_ratio": 2.5,  # was 1.5
    },
    "snapshot_rows": [
        {
            "timestamp": NOW_ISO,
            "symbol": "BTC",
            "expiry": EXPIRY_1_ISO,
            "raw_value": 0.12,  # was 0.08
            "market_value": 0.05,
        },
    ],
}

r = client.patch("/api/blocks/manual_btc_vol_spike", json=UPDATE_PAYLOAD)
assert r.status_code == 200, f"Update block failed: {r.status_code} {r.text}"
updated = r.json()
print(f"Block updated:")
print(f"  var_fair_ratio: {updated['var_fair_ratio']}  (was 1.5)")
print(f"  raw_value:      {updated['raw_value']}  (was 0.08)")
assert updated["var_fair_ratio"] == 2.5
assert updated["raw_value"] == 0.12
print("\n✓ Block update succeeded")

## 14. Transform Configuration

Inspect the available transform steps and their parameter schemas, then update
a parameter to verify the pipeline re-runs with the new configuration.

In [ ]:
# GET /api/transforms — inspect all pipeline steps
r = client.get("/api/transforms")
assert r.status_code == 200
steps = r.json()["steps"]
print(f"Pipeline steps: {len(steps)}")

for step_name, step in steps.items():
    selected = step["selected"]
    n_transforms = len(step["transforms"])
    params = step["params"]
    formula = next((t["formula"] for t in step["transforms"] if t["name"] == selected), "")
    print(f"\n  {step_name}")
    print(f"    label:      {step['label']}")
    print(f"    selected:   {selected}")
    print(f"    available:  {[t['name'] for t in step['transforms']]}")
    print(f"    params:     {params}")
    if formula:
        print(f"    formula:    {formula}")

# Validate each transform has the expected schema fields
for step_name, step in steps.items():
    for t in step["transforms"]:
        assert "name" in t and "description" in t and "params" in t
        for p in t["params"]:
            assert "name" in p and "type" in p and "default" in p

print(f"\n✓ {len(steps)} transform steps inspected")

In [ ]:
# PATCH /api/transforms — update a transform parameter
# Find a step with a tunable float param to modify
patch_body = {}
patched_step = None
patched_param = None
original_value = None

for step_name, step in steps.items():
    selected_transform = next(
        (t for t in step["transforms"] if t["name"] == step["selected"]), None
    )
    if selected_transform:
        for p in selected_transform["params"]:
            if p["type"] == "float" and p.get("min") is not None:
                patched_step = step_name
                patched_param = p["name"]
                original_value = step["params"].get(p["name"], p["default"])
                # Nudge the value slightly
                new_value = original_value * 1.1 if original_value != 0 else 0.5
                patch_body[f"{step_name}_params"] = {p["name"]: new_value}
                break
    if patch_body:
        break

if patch_body:
    print(f"Patching {patched_step}.{patched_param}: {original_value} → {new_value}")
    r = client.patch("/api/transforms", json=patch_body)
    assert r.status_code == 200, f"Transform patch failed: {r.status_code} {r.text}"
    updated_steps = r.json()["steps"]
    updated_val = updated_steps[patched_step]["params"].get(patched_param)
    print(f"After patch: {patched_param} = {updated_val}")

    # Restore original
    restore_body = {f"{patched_step}_params": {patched_param: original_value}}
    r = client.patch("/api/transforms", json=restore_body)
    assert r.status_code == 200
    print(f"Restored: {patched_param} = {original_value}")
    print("\n✓ Transform param update + restore succeeded")
else:
    print("No tunable float param found — skipping transform patch test")

## 15. Snapshot Update

Send updated snapshot values and verify the pipeline re-runs with new data.

In [ ]:
# Bump rv raw_value to see a change in the pipeline output
r = client.post("/api/snapshots", json={
    "stream_name": "rv",
    "rows": [
        {"timestamp": NOW_ISO, "symbol": "BTC", "expiry": EXPIRY_1_ISO, "raw_value": 0.55, "market_value": 0.50},
        {"timestamp": NOW_ISO, "symbol": "BTC", "expiry": EXPIRY_2_ISO, "raw_value": 0.55, "market_value": 0.50},
        {"timestamp": NOW_ISO, "symbol": "ETH", "expiry": EXPIRY_1_ISO, "raw_value": 0.70, "market_value": 0.65},
        {"timestamp": NOW_ISO, "symbol": "ETH", "expiry": EXPIRY_2_ISO, "raw_value": 0.70, "market_value": 0.65},
    ],
})
assert r.status_code == 200
data = r.json()
assert data["pipeline_rerun"] is True
print(f"rv update: {data['rows_accepted']} rows, pipeline_rerun={data['pipeline_rerun']}")
print("✓ Snapshot update triggered pipeline re-run")

## 16. Client WebSocket — Snapshot Frames

Send snapshot data through the authenticated `/ws/client` channel using a READY stream
(not just `__test__`). Verify ACKs include `pipeline_rerun: true`.

In [ ]:
async def test_client_ws_snapshots():
    """Send snapshot frames on a real READY stream via /ws/client."""
    async with websockets.connect(CLIENT_WS_URL) as ws:
        print(f"Connected to /ws/client\n")

        # Catch-up payload
        try:
            catchup = await asyncio.wait_for(ws.recv(), timeout=5.0)
            data = json.loads(catchup)
            n_pos = len(data.get("positions", []))
            print(f"Catch-up: {n_pos} positions\n")
        except asyncio.TimeoutError:
            print("No catch-up payload\n")

        # -- Send __test__ stream frame (connectivity check) --
        frame_test = {
            "seq": 1,
            "stream_name": "__test__",
            "rows": [{"timestamp": NOW_ISO, "raw_value": 0.99, "symbol": "BTC"}],
        }
        await ws.send(json.dumps(frame_test))
        ack = json.loads(await asyncio.wait_for(ws.recv(), timeout=5.0))
        assert ack["type"] == "ack" and ack["seq"] == 1
        print(f"Frame 1 (__test__): ACK rows_accepted={ack['rows_accepted']}")

        # -- Send rv stream frame (should trigger pipeline rerun) --
        frame_rv = {
            "seq": 2,
            "stream_name": "rv",
            "rows": [
                {"timestamp": NOW_ISO, "symbol": "BTC", "expiry": EXPIRY_1_ISO, "raw_value": 0.58},
                {"timestamp": NOW_ISO, "symbol": "ETH", "expiry": EXPIRY_1_ISO, "raw_value": 0.72},
            ],
        }
        await ws.send(json.dumps(frame_rv))
        ack = json.loads(await asyncio.wait_for(ws.recv(), timeout=10.0))
        assert ack["type"] == "ack" and ack["seq"] == 2
        print(f"Frame 2 (rv):      ACK rows_accepted={ack['rows_accepted']}  pipeline_rerun={ack['pipeline_rerun']}")
        assert ack["pipeline_rerun"] is True, "Expected pipeline_rerun=true for real stream"

        # -- Listen for 2 position broadcasts to confirm ticker is running --
        ticks_received = 0
        while ticks_received < 2:
            raw = await asyncio.wait_for(ws.recv(), timeout=10.0)
            msg = json.loads(raw)
            if "type" in msg:
                continue  # skip ACKs
            positions = msg.get("positions", [])
            print(f"Tick {ticks_received}: {len(positions)} positions")
            ticks_received += 1

        print("\n✓ Client WS snapshot frames work (ACK + pipeline rerun + broadcasts)")

await test_client_ws_snapshots()

## 17. Client WebSocket — Market Value Frames

Send `type: "market_value"` frames on `/ws/client`. These set the dirty flag for
coalesced pipeline reruns on the next WS tick (no immediate rerun).

In [ ]:
async def test_client_ws_market_values():
    """Send market_value frames via /ws/client and verify ACK."""
    async with websockets.connect(CLIENT_WS_URL) as ws:
        print(f"Connected to /ws/client\n")

        # Consume catch-up
        try:
            await asyncio.wait_for(ws.recv(), timeout=5.0)
        except asyncio.TimeoutError:
            pass

        # Send a market_value frame
        mv_frame = {
            "type": "market_value",
            "seq": 100,
            "entries": [
                {"symbol": "BTC", "expiry": EXPIRY_1_ISO, "total_vol": 0.60},
                {"symbol": "ETH", "expiry": EXPIRY_1_ISO, "total_vol": 0.75},
            ],
        }
        await ws.send(json.dumps(mv_frame))
        ack = json.loads(await asyncio.wait_for(ws.recv(), timeout=5.0))
        print(f"Market value frame ACK: {ack}")
        assert ack["type"] == "ack"
        assert ack["seq"] == 100
        assert ack["rows_accepted"] == 2
        # Market value frames don't trigger immediate pipeline rerun (dirty flag instead)
        assert ack["pipeline_rerun"] is False

        # Verify via HTTP that the values were stored
        # (Small delay to let the WS handler finish writing)
        await asyncio.sleep(0.5)

        print("\n✓ Market value WS frames accepted (dirty-flag coalescing)")

await test_client_ws_market_values()

# Verify via HTTP
r = client.get("/api/market-values")
entries = r.json()["entries"]
btc_entry = next((e for e in entries if e["symbol"] == "BTC" and e["expiry"] == EXPIRY_1_ISO), None)
print(f"\nHTTP verify — BTC/{EXPIRY_1_ISO[:10]} total_vol: {btc_entry['total_vol'] if btc_entry else 'NOT FOUND'}")
if btc_entry:
    assert btc_entry["total_vol"] == 0.60, f"Expected 0.60, got {btc_entry['total_vol']}"
    print("✓ Market values written by WS are visible via HTTP")

## 18. LLM Investigation (SSE)

Test the `/api/investigate` endpoint which streams LLM tokens via SSE.
Skipped if no `OPENROUTER_API_KEY` is configured on the server (returns 503).

In [ ]:
# POST /api/investigate — SSE streaming chat
# Test each mode: investigate, build, general

LLM_MODES = [
    {
        "mode": "investigate",
        "conversation": [{"role": "user", "content": "Why did BTC desired position change?"}],
        "cell_context": {"type": "position", "symbol": "BTC", "expiry": EXPIRY_1_ISO},
    },
    {
        "mode": "build",
        "conversation": [{"role": "user", "content": "I want to add a realized vol stream for ETH"}],
    },
    {
        "mode": "general",
        "conversation": [{"role": "user", "content": "What is the current bankroll?"}],
    },
]

for test in LLM_MODES:
    payload = {
        "conversation": test["conversation"],
        "mode": test["mode"],
    }
    if "cell_context" in test:
        payload["cell_context"] = test["cell_context"]

    with client.stream("POST", "/api/investigate", json=payload) as r:
        if r.status_code == 503:
            print(f"  [{test['mode']:12s}]  SKIPPED — no OPENROUTER_API_KEY")
            continue

        assert r.status_code == 200, f"LLM {test['mode']} failed: {r.status_code}"
        assert r.headers.get("content-type", "").startswith("text/event-stream")

        tokens = []
        done = False
        for line in r.iter_lines():
            if line.startswith("data: "):
                data = line[6:]
                if data == "[DONE]":
                    done = True
                    break
                tokens.append(data)
            elif line.startswith("event: error"):
                # Next data line will have the error
                continue

        preview = "".join(json.loads(t) for t in tokens[:5] if t) if tokens else "(empty)"
        print(f"  [{test['mode']:12s}]  {len(tokens)} tokens  done={done}  preview: {preview[:80]}...")

print("\n✓ LLM investigation SSE tested")

## 19. WS Position Subscription

Subscribe to `/ws` and verify the desired-position broadcast payload shape.
Validates that positions contain the expected fields from `DesiredPosition` type.

In [ ]:
POSITION_FIELDS = [
    "symbol", "expiry", "edge", "smoothedEdge", "variance", "smoothedVar",
    "desiredPos", "rawDesiredPos", "currentPos", "totalFair", "totalMarketFair",
    "changeMagnitude", "updatedAt",
]
PAYLOAD_FIELDS = ["streams", "context", "positions", "updates"]

async def test_ws_subscription(max_ticks=3):
    """Subscribe to /ws and validate payload shape."""
    async with websockets.connect(WS_URL) as ws:
        print(f"Connected to {WS_URL}\n")
        tick = 0
        while tick < max_ticks:
            raw = await asyncio.wait_for(ws.recv(), timeout=10.0)
            data = json.loads(raw)

            # Validate top-level payload shape
            for field in PAYLOAD_FIELDS:
                assert field in data, f"Missing payload field: {field}"

            positions = data["positions"]
            context = data["context"]
            streams = data["streams"]
            updates = data["updates"]

            # Context must have lastUpdateTimestamp
            assert "lastUpdateTimestamp" in context

            print(f"Tick {tick}: {len(positions)} positions, {len(streams)} streams, {len(updates)} updates")

            # Validate position shape
            for p in positions:
                for field in POSITION_FIELDS:
                    assert field in p, f"Position missing field: {field}"
                print(
                    f"  {p['symbol']:>5s}  {p['expiry']:>9s}  "
                    f"desired={p['desiredPos']:>10.2f}  "
                    f"edge={p['edge']:.6f}  var={p['variance']:.6f}"
                )

            tick += 1

        print(f"\n✓ {max_ticks} ticks received with valid payload shape")

await test_ws_subscription()

## 20. Error Cases

Verify that the API returns correct error codes for invalid operations.

In [ ]:
errors_tested = 0

# -- Duplicate stream creation → 409 --
r = client.post("/api/streams", json={"stream_name": "rv", "key_cols": ["symbol"]})
assert r.status_code == 409, f"Expected 409 for duplicate, got {r.status_code}"
print(f"  Duplicate stream:          {r.status_code} ✓")
errors_tested += 1

# -- Snapshot to non-existent stream → 404 --
r = client.post("/api/snapshots", json={
    "stream_name": "nonexistent",
    "rows": [{"timestamp": NOW_ISO, "raw_value": 1.0}],
})
assert r.status_code == 404, f"Expected 404, got {r.status_code}"
print(f"  Snapshot to missing stream: {r.status_code} ✓")
errors_tested += 1

# -- Snapshot to PENDING stream → 422 --
# Create a temp stream (PENDING), try to ingest, then clean up
client.post("/api/streams", json={"stream_name": "temp_pending", "key_cols": ["symbol"]})
r = client.post("/api/snapshots", json={
    "stream_name": "temp_pending",
    "rows": [{"timestamp": NOW_ISO, "raw_value": 1.0, "symbol": "BTC"}],
})
assert r.status_code == 422, f"Expected 422 for PENDING stream, got {r.status_code}"
print(f"  Snapshot to PENDING stream: {r.status_code} ✓")
client.delete("/api/streams/temp_pending")  # clean up
errors_tested += 1

# -- Configure non-existent stream → 404 --
r = client.post("/api/streams/nonexistent/configure", json={
    "scale": 1.0, "offset": 0.0, "exponent": 1.0, "block": {},
})
assert r.status_code == 404, f"Expected 404, got {r.status_code}"
print(f"  Configure missing stream:  {r.status_code} ✓")
errors_tested += 1

# -- Delete non-existent stream → 404 --
r = client.delete("/api/streams/nonexistent")
assert r.status_code == 404, f"Expected 404, got {r.status_code}"
print(f"  Delete missing stream:     {r.status_code} ✓")
errors_tested += 1

# -- Update non-existent block → 404 --
r = client.patch("/api/blocks/nonexistent", json={"scale": 2.0})
assert r.status_code == 404, f"Expected 404, got {r.status_code}"
print(f"  Update missing block:      {r.status_code} ✓")
errors_tested += 1

# -- Delete non-existent market value → 404 --
r = client.delete("/api/market-values/FAKE/2099-01-01T00:00:00")
assert r.status_code == 404, f"Expected 404, got {r.status_code}"
print(f"  Delete missing MV:         {r.status_code} ✓")
errors_tested += 1

# -- Pipeline timeseries for non-existent dimension → 404 --
r = client.get("/api/pipeline/timeseries", params={"symbol": "FAKE", "expiry": "2099-01-01T00:00:00"})
assert r.status_code == 404, f"Expected 404, got {r.status_code}"
print(f"  Timeseries missing dim:    {r.status_code} ✓")
errors_tested += 1

# -- Invalid expiry format → 422 --
r = client.get("/api/pipeline/timeseries", params={"symbol": "BTC", "expiry": "not-a-date"})
assert r.status_code == 422, f"Expected 422, got {r.status_code}"
print(f"  Invalid expiry format:     {r.status_code} ✓")
errors_tested += 1

# -- Snapshot missing required columns → 422 --
r = client.post("/api/snapshots", json={
    "stream_name": "rv",
    "rows": [{"timestamp": NOW_ISO, "raw_value": 1.0}],  # missing symbol, expiry
})
assert r.status_code == 422, f"Expected 422, got {r.status_code}"
print(f"  Snapshot missing columns:  {r.status_code} ✓")
errors_tested += 1

print(f"\n✓ {errors_tested} error cases validated")

## 21. Cleanup

Delete all streams created during this test run. Verify the server returns to its initial state.

In [ ]:
# Delete all streams we created (not __test__)
CREATED_STREAMS = ["rv", "mean_iv", "events", "manual_btc_vol_spike"]
for name in CREATED_STREAMS:
    r = client.delete(f"/api/streams/{name}")
    if r.status_code == 204:
        print(f"  Deleted: {name}")
    elif r.status_code == 404:
        print(f"  Already gone: {name}")
    else:
        print(f"  Unexpected: {name} → {r.status_code}")

# Verify only __test__ remains
r = client.get("/api/streams")
remaining = r.json()["streams"]
remaining_names = [s["stream_name"] for s in remaining]
print(f"\nRemaining streams: {remaining_names}")
assert remaining_names == ["__test__"], f"Expected only __test__, got {remaining_names}"

print("\n✓ Cleanup complete — server back to initial state")
print("\n" + "=" * 60)
print("  ALL TESTS PASSED")
print("=" * 60)